# Cross-model behaviour-direction transfer

Six LLM-judge behaviour directions are extracted from DeepSeek-R1-Distill-Qwen-7B
at layer 13 and projected into the activation space of Qwen2.5-7B-Instruct.
Diagnostic Spearman correlations and active-steering null results back the
appendix subsection `\label{app:cross_model_steering}` (Table `\label{tab:cross_model_diag}`).

In [1]:
import os, json
from pathlib import Path

_root = Path.cwd()
while not (_root / 'pyproject.toml').exists() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

RESULTS = Path(os.environ.get('IRT_RESULTS_ROOT', 'data/results'))
XMODEL = RESULTS / 'code' / 'qwen-7b' / 'cross_model_steering'

ALPHAS = [-3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]
BEHAVIOURS = [
    'strategy_shifting',
    'uncertainty_monitoring',
    'self_correction',
    'verification',
    'problem_restatement',
    'subgoal_decomposition',
]

## Diagnostic correlations (Table `\label{tab:cross_model_diag}`)

For each behaviour direction extracted from R1-7b and projected into Qwen-7b,
compute Spearman $\rho$ between the projection score and the corresponding
behaviour rate in Qwen-7b traces. Source: `cross_model_diagnostic.json`.

In [2]:
with open(XMODEL / 'cross_model_diagnostic.json') as f:
    diag = json.load(f)

dp = diag['diagnostic_projections']
projections = {item['direction']: item for item in dp}

rows = []
for behaviour in BEHAVIOURS:
    direction = f'{behaviour}_lengthonly'
    rho_key = f'rho_{behaviour}'
    p_key = f'p_{behaviour}'
    if direction not in projections or rho_key not in projections[direction]:
        rows.append({'behaviour': behaviour, 'rho_match': np.nan, 'p_match': np.nan,
                     'rho_difficulty': np.nan})
        continue
    rec = projections[direction]
    rows.append({
        'behaviour': behaviour,
        'rho_match': float(rec[rho_key]),
        'p_match': float(rec[p_key]),
        'rho_difficulty': float(rec['rho_difficulty']),
    })
diag_df = pd.DataFrame(rows)
print(diag_df.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))

             behaviour  rho_match   p_match  rho_difficulty
     strategy_shifting     0.0312    0.1194          0.0183
uncertainty_monitoring    -0.0017    0.9308         -0.0199
       self_correction     0.0039    0.8454         -0.0350
          verification    -0.0186    0.3535         -0.0507
   problem_restatement    -0.0529    0.0082         -0.0781
 subgoal_decomposition     0.1601    0.0000          0.1317


In [3]:
BEHAVIOUR_LATEX = {
    'strategy_shifting': 'Strategy shifting',
    'uncertainty_monitoring': 'Uncertainty monitoring',
    'self_correction': 'Self-correction',
    'verification': 'Verification',
    'problem_restatement': 'Problem restatement',
    'subgoal_decomposition': 'Subgoal decomposition',
}

def fmt_p(p):
    if p < 1e-15:
        return r'$<10^{-15}$'
    if p < 1e-3:
        exp = int(np.floor(np.log10(p)))
        return f'${p / 10**exp:.1f}\\times 10^{{{exp}}}$'
    return f'{p:.3f}'

lines = [
    r'\begin{tabular}{@{}lrr@{}}',
    r'\toprule',
    r'Transferred direction & $\rho$ vs target behaviour rate & $p$ \\',
    r'\midrule',
]
for _, r in diag_df.iterrows():
    lines.append(
        f'{BEHAVIOUR_LATEX[r["behaviour"]]} & {r["rho_match"]:+.3f} & {fmt_p(r["p_match"])} \\\\'
    )
lines += [r'\bottomrule', r'\end{tabular}']
print('\n'.join(lines))

\begin{tabular}{@{}lrr@{}}
\toprule
Transferred direction & $\rho$ vs target behaviour rate & $p$ \\
\midrule
Strategy shifting & +0.031 & 0.119 \\
Uncertainty monitoring & -0.002 & 0.931 \\
Self-correction & +0.004 & 0.845 \\
Verification & -0.019 & 0.353 \\
Problem restatement & -0.053 & 0.008 \\
Subgoal decomposition & +0.160 & $<10^{-15}$ \\
\bottomrule
\end{tabular}


## Active steering: null effect on $\rho_\perp^D$, accuracy, reasoning length, behaviour rates

For each of the four behaviour directions with corresponding steering parquets
(self-correction, verification, strategy shifting, uncertainty monitoring),
group by $\alpha$ and check that the per-direction summary statistics are
constant across $\alpha$ within sampling noise. No figure is produced.

In [4]:
def aggregate_per_problem(df, y_col='directness_reasoning', len_col='reasoning_length'):
    df = df.dropna(subset=[y_col, len_col, 'difficulty']).copy()
    df = df[df[len_col] > 0]
    agg = df.groupby('problem_id').agg(
        directness=(y_col, 'mean'),
        reasoning_length=(len_col, 'mean'),
        difficulty=('difficulty', 'first'),
    ).reset_index()
    agg['log_reasoning_length'] = np.log(agg['reasoning_length'])
    return agg


def rho_perp_one_sided(agg):
    if len(agg) < 10:
        return np.nan
    X = np.column_stack([np.ones(len(agg)), agg['log_reasoning_length'].values])
    beta, *_ = np.linalg.lstsq(X, agg['directness'].values, rcond=None)
    resid_d = agg['directness'].values - X @ beta
    rho, _ = spearmanr(resid_d, agg['difficulty'].values)
    return float(rho)


steered_behaviours = ['self_correction', 'strategy_shifting',
                      'uncertainty_monitoring', 'verification']

per_alpha = []
for behaviour in steered_behaviours:
    p = XMODEL / f'cross_model_{behaviour}_lengthonly_pooled.parquet'
    df = pd.read_parquet(p)
    for alpha in ALPHAS:
        sub = df[df['alpha'] == alpha]
        agg = aggregate_per_problem(sub)
        per_alpha.append({
            'behaviour': behaviour,
            'alpha': alpha,
            'rho_perp': rho_perp_one_sided(agg),
            'mean_reasoning_length': float(sub['reasoning_length'].mean()),
            'mean_correct': float(sub['correct'].mean()) if 'correct' in sub.columns else np.nan,
            'n_problems': int(agg.shape[0]),
        })

steer_df = pd.DataFrame(per_alpha)
print(steer_df.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))

             behaviour     alpha  rho_perp  mean_reasoning_length  mean_correct  n_problems
       self_correction   -3.0000   -0.2403               965.0733        0.0933         150
       self_correction   -2.0000   -0.2026               956.9467        0.0933         150
       self_correction   -1.0000   -0.3444               971.7067        0.1000         150
       self_correction   -0.5000   -0.2680               979.2067        0.1200         150
       self_correction    0.0000   -0.2698              1021.1933        0.0733         150
       self_correction    0.5000   -0.2770              1007.8933        0.0867         150
       self_correction    1.0000   -0.2623               977.1200        0.0867         150
       self_correction    2.0000   -0.2436              1010.8333        0.0800         150
       self_correction    3.0000   -0.2077              1031.3000        0.0733         150
     strategy_shifting   -3.0000   -0.2044              1000.9000        0.0533 

In [5]:
# Range summaries per behaviour: confirm rho_perp, reasoning length, and accuracy
# do not vary with alpha beyond sampling noise.
summary = (
    steer_df.groupby('behaviour')
    .agg(
        rho_min=('rho_perp', 'min'),
        rho_max=('rho_perp', 'max'),
        rho_range=('rho_perp', lambda s: s.max() - s.min()),
        rl_min_k=('mean_reasoning_length', lambda s: s.min() / 1000),
        rl_max_k=('mean_reasoning_length', lambda s: s.max() / 1000),
        acc_min=('mean_correct', 'min'),
        acc_max=('mean_correct', 'max'),
    )
    .round(3)
)
print(summary.to_string())

                        rho_min  rho_max  rho_range  rl_min_k  rl_max_k  acc_min  acc_max
behaviour                                                                                
self_correction          -0.344   -0.203      0.142     0.957     1.031    0.073    0.120
strategy_shifting        -0.284   -0.125      0.159     0.976     1.020    0.053    0.120
uncertainty_monitoring   -0.371   -0.177      0.194     0.989     1.020    0.067    0.133
verification             -0.337   -0.214      0.123     0.982     1.042    0.040    0.127


## Summary

- Five of six diagnostic correlations are statistically indistinguishable from
  zero ($|\rho| < 0.04$, $p > 0.1$), with problem restatement weakly significant
  but with negligible effect size ($\rho = -0.053$, $p \approx 8\times 10^{-3}$).
- Subgoal decomposition is the lone positive transfer ($\rho = 0.160$,
  $p < 10^{-15}$); the same direction also correlates with item difficulty
  ($\rho = 0.132$).
- Active steering at nine $\alpha$ values along the four primary behaviour
  directions produces no monotonic dose-response in $\rho_\perp^D$, accuracy,
  or reasoning length.